# Problem-statement localization hints

For each bug-fix task, this notebook derives three signals from the problem statement and gold patch:

- **Gold path:** a complete repository-relative path modified by the gold patch appears in the statement.
- **Gold object name:** the gold AL filename without its object-type suffix (for example, `SalesHeader.Table.al` → `SalesHeader`) appears after spaces and punctuation are ignored. This is an approximate indicator, not a contamination label.
- **Call stack:** the statement contains “call stack” or “stack trace”.

In [1]:
import re
from pathlib import PurePosixPath

import pandas as pd

from bcbench.collection.patch_utils import extract_file_paths_from_patch
from bcbench.config import get_config
from bcbench.dataset import BugFixEntry

_config = get_config()
_object_type_suffix = re.compile(r"\.(table|page|codeunit|report|xmlport|query|enum|interface|permissionset|profile|controladdin|dotnet)\.al$", re.IGNORECASE)


def contains_gold_path(statement: str, gold_paths: list[str]) -> bool:
    normalized_statement = statement.replace("\\", "/").casefold()
    return any(path.replace("\\", "/").casefold() in normalized_statement for path in gold_paths)


def mentions_gold_object_name(statement: str, gold_paths: list[str]) -> bool:
    normalized_statement = re.sub(r"[^a-z0-9]", "", statement.casefold())
    object_names = [re.sub(r"[^a-z0-9]", "", _object_type_suffix.sub("", PurePosixPath(path).name).casefold()) for path in gold_paths]
    return any(len(name) >= 8 and name in normalized_statement for name in object_names)


entries: list[BugFixEntry] = BugFixEntry.load(_config.paths.dataset_dir / "bcbench.jsonl")
dataset_df = pd.DataFrame(
    [
        {
            "instance_id": entry.instance_id,
            "contains_gold_path": contains_gold_path(entry.get_task(), extract_file_paths_from_patch(entry.patch)),
            "mentions_gold_object_name": mentions_gold_object_name(entry.get_task(), extract_file_paths_from_patch(entry.patch)),
            "contains_call_stack": bool(re.search(r"call\s*stack|stack\s*trace", entry.get_task(), re.IGNORECASE)),
        }
        for entry in entries
    ]
)
print(f"Loaded {len(dataset_df)} bug-fix tasks")

Loaded 101 bug-fix tasks


In [2]:
signals = {
    "contains_gold_path": "Complete gold path",
    "mentions_gold_object_name": "Gold object name",
    "contains_call_stack": "Call stack",
}
signal_stats = pd.DataFrame([{"Signal": label, "Tasks": int(dataset_df[column].sum()), "Percentage": round(dataset_df[column].mean() * 100, 1)} for column, label in signals.items()])
print(signal_stats.to_string(index=False))
dataset_df.loc[dataset_df["contains_gold_path"], ["instance_id"]]

            Signal  Tasks  Percentage
Complete gold path      1         1.0
  Gold object name     28        27.7
        Call stack      7         6.9


,instance_id
34,microsoft__BCApps-4699


## Findings

- Only 1 of 101 tasks (1.0%), `microsoft__BCApps-4699`, contains a complete gold path, so direct path disclosure cannot explain a broad benchmark-level contamination signal.
- Gold object names occur in 28 tasks (27.7%) and call stacks in 7 (6.9%), so genuine deduction from the statement is a meaningful confounder for context-free file localization.